# Retrieve and prepare decoded replay packets

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joyalzzy/playable-replays/blob/ml/replay-packets-download.ipynb)

This Google Colab-only notebook retrieves one revision-pinned batch from [`maknee/league-of-legends-decoded-replay-packets`](https://huggingface.co/datasets/maknee/league-of-legends-decoded-replay-packets), verifies its published SHA-256, removes likely account identifiers, compacts bounded packet sequences, and exports training-ready JSONL for [`player-ai.ipynb`](./player-ai.ipynb). It refuses to download data outside Colab and does not train a model or connect to the product runtime.

The dataset publisher declares Apache-2.0 and states that the work is not endorsed by Riot Games. Packet-native `x`/`z` values remain dataset coordinates, not normalized simulator coordinates.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import platform
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if not IN_COLAB:
    raise RuntimeError("Training-data download is restricted to Google Colab; open this notebook in Colab")

def probe(url: str) -> dict[str, object]:
    request = Request(url, headers={"User-Agent": "playable-replays-dataset-check/1.0"})
    try:
        with urlopen(request, timeout=12) as response:
            return {"reachable": True, "status": response.status, "finalUrl": response.url}
    except HTTPError as error:
        return {"reachable": True, "status": error.code, "finalUrl": error.url}
    except (URLError, TimeoutError, OSError) as error:
        return {"reachable": False, "error": type(error).__name__}

runtime_report = {
    "inColab": IN_COLAB,
    "python": platform.python_version(),
    "datasetConnectivity": probe("https://huggingface.co/datasets/maknee/league-of-legends-decoded-replay-packets"),
}
print(json.dumps(runtime_report, indent=2))
print("Colab-only data retrieval gate passed.")

## 1. Retrieval limits and immutable source

The full repository is intentionally not downloaded. The default batch is approximately 86.6 MB compressed. Oversized individual games are skipped with bounded reads instead of allowing one JSONL line to consume unbounded memory.

In [ ]:
from pathlib import Path

HF_DATASET_REPO = "maknee/league-of-legends-decoded-replay-packets"
HF_DATASET_REVISION = "04f9c7350e9ffcc689b731875ad9baf3ff6eaa6d"
HF_REPLAY_FILE = "13_2/batch_001.jsonl.gz"
HF_REPLAY_SHA256 = "c4b8846d8b088481afee92ad9d660fc959532438e64b9616b09a1690caf7e6e7"
MAX_DOWNLOAD_BYTES = 120_000_000
MAX_SOURCE_LINES = 64  # @param {type:"integer"}
MAX_GAME_LINE_BYTES = 64_000_000  # @param {type:"integer"}
MAX_REPLAY_GAMES = 8  # @param {type:"integer"}
MAX_PACKETS_PER_GAME = 2048  # @param {type:"integer"}
PACKET_SAMPLE_STRIDE = 4  # @param {type:"integer"}
OUTPUT_DIR = Path("/content/playable-replays-output")
OUTPUT_RECORDS = OUTPUT_DIR / "decoded-replay-records.jsonl"
OUTPUT_MANIFEST = OUTPUT_DIR / "decoded-replay-manifest.json"

if min(MAX_SOURCE_LINES, MAX_GAME_LINE_BYTES, MAX_REPLAY_GAMES, MAX_PACKETS_PER_GAME, PACKET_SAMPLE_STRIDE) < 1:
    raise ValueError("All retrieval limits must be positive")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Prepared-record output: {OUTPUT_RECORDS.resolve()}")

## 2. Download and verify the pinned batch

The cache is reused only after a complete SHA-256 match. Partial or mismatched files never become the accepted input. No Hugging Face token is required for this public file.

In [ ]:
import hashlib
from urllib.parse import quote

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def download_replay_batch() -> Path:
    if not IN_COLAB:
        raise RuntimeError("Replay data may only be downloaded inside Google Colab")
    cache_path = OUTPUT_DIR / "hf-cache" / HF_DATASET_REVISION / HF_REPLAY_FILE
    if cache_path.exists():
        actual_hash = sha256_file(cache_path)
        if actual_hash != HF_REPLAY_SHA256:
            raise ValueError(f"Cached checksum mismatch at {cache_path}; remove that file manually before retrying")
        print(f"Using verified cached batch: {cache_path}")
        return cache_path
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    encoded_file = quote(HF_REPLAY_FILE, safe="/")
    url = f"https://huggingface.co/datasets/{HF_DATASET_REPO}/resolve/{HF_DATASET_REVISION}/{encoded_file}?download=true"
    request = Request(url, headers={"User-Agent": "playable-replays-offline-dataset/1.0"})
    partial_path = cache_path.with_name(cache_path.name + ".part")
    total_bytes = 0
    print(f"Downloading pinned batch: {HF_REPLAY_FILE}")
    with urlopen(request, timeout=60) as response, partial_path.open("wb") as output:
        while chunk := response.read(1024 * 1024):
            total_bytes += len(chunk)
            if total_bytes > MAX_DOWNLOAD_BYTES:
                raise ValueError("Dataset batch exceeded the configured download cap")
            output.write(chunk)
    actual_hash = sha256_file(partial_path)
    if actual_hash != HF_REPLAY_SHA256:
        raise ValueError(f"Downloaded checksum mismatch: {actual_hash}")
    partial_path.replace(cache_path)
    print(f"Verified {total_bytes:,} bytes with SHA-256 {actual_hash}")
    return cache_path

replay_batch_path = download_replay_batch()

## 3. Compact packet transitions and export prepared records

Each exported record contains a bounded current packet and the next sampled packet as its factual target. Consecutive packets with the same dataset-native `time` share an ordinal `tick`; a new ordinal tick starts whenever the timestamp changes. The source packet array and `state.packetIndex` remain authoritative because some decoded games contain timestamp regressions. Regressions are counted in the manifest instead of reordering or rejecting packets. Stable game IDs derive only from dataset provenance and source-line position. Generic `name` plus likely account/summoner identifier fields are omitted.

In [ ]:
import gzip
import math
from typing import Any, BinaryIO, Iterator

MAX_PACKET_DEPTH = 4
MAX_PACKET_MAPPING_ITEMS = 24
MAX_PACKET_LIST_ITEMS = 16
MAX_PACKET_STRING_LENGTH = 160
SENSITIVE_KEY_FRAGMENTS = ("summoner", "puuid", "account", "player_name", "riot_id")

def bounded_jsonl_lines(handle: BinaryIO, max_bytes: int) -> Iterator[tuple[int, bytes | None]]:
    source_line = 0
    while True:
        chunk = handle.readline(max_bytes + 1)
        if not chunk:
            return
        source_line += 1
        if len(chunk) <= max_bytes:
            yield source_line, chunk
            continue
        while chunk and not chunk.endswith(b"\n"):
            chunk = handle.readline(max_bytes + 1)
        yield source_line, None

def compact_packet_value(value: Any, depth: int = 0) -> Any:
    if value is None or isinstance(value, (bool, int)):
        return value
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, str):
        return value[:MAX_PACKET_STRING_LENGTH]
    if depth >= MAX_PACKET_DEPTH:
        return "<depth-truncated>"
    if isinstance(value, dict):
        compacted: dict[str, Any] = {}
        for raw_key in sorted(value, key=lambda item: str(item)):
            key = str(raw_key)
            lowered = key.lower()
            if lowered == "name" or any(fragment in lowered for fragment in SENSITIVE_KEY_FRAGMENTS):
                continue
            compacted[key[:80]] = compact_packet_value(value[raw_key], depth + 1)
            if len(compacted) >= MAX_PACKET_MAPPING_ITEMS:
                break
        if len(value) > len(compacted):
            compacted["_truncatedOrRedactedItems"] = len(value) - len(compacted)
        return compacted
    if isinstance(value, list):
        compacted = [compact_packet_value(item, depth + 1) for item in value[:MAX_PACKET_LIST_ITEMS]]
        if len(value) > MAX_PACKET_LIST_ITEMS:
            compacted.append({"_truncatedItems": len(value) - MAX_PACKET_LIST_ITEMS})
        return compacted
    return repr(value)[:MAX_PACKET_STRING_LENGTH]

def compact_event(event: Any, game_index: int, packet_index: int) -> dict[str, Any]:
    if not isinstance(event, dict) or len(event) != 1:
        raise ValueError(f"game {game_index} packet {packet_index}: expected exactly one packet-type key")
    packet_type, payload = next(iter(event.items()))
    if not isinstance(packet_type, str) or not packet_type:
        raise ValueError(f"game {game_index} packet {packet_index}: invalid packet type")
    return {"packetType": packet_type, "payload": compact_packet_value(payload)}

def packet_time_seconds(event: Any, game_index: int, packet_index: int) -> float:
    if not isinstance(event, dict) or len(event) != 1:
        raise ValueError(f"game {game_index} packet {packet_index}: expected exactly one packet-type key")
    payload = next(iter(event.values()))
    raw_time = payload.get("time") if isinstance(payload, dict) else None
    if not isinstance(raw_time, (int, float)) or isinstance(raw_time, bool) or not math.isfinite(raw_time) or raw_time < 0:
        raise ValueError(f"game {game_index} packet {packet_index}: time must be a finite non-negative number")
    return float(raw_time)

def packet_ticks(events: list[Any], game_index: int, packet_limit: int) -> tuple[list[int], list[float], int]:
    ticks: list[int] = []
    times: list[float] = []
    current_tick = -1
    previous_time: float | None = None
    timestamp_regressions = 0
    for packet_index in range(packet_limit):
        packet_time = packet_time_seconds(events[packet_index], game_index, packet_index)
        if previous_time is not None and packet_time < previous_time:
            timestamp_regressions += 1
        if previous_time is None or packet_time != previous_time:
            current_tick += 1
        ticks.append(current_tick)
        times.append(packet_time)
        previous_time = packet_time
    return ticks, times, timestamp_regressions

def records_from_replay_batch(path: Path) -> tuple[list[dict[str, Any]], dict[str, int]]:
    dataset_url = f"https://huggingface.co/datasets/{HF_DATASET_REPO}"
    converted: list[dict[str, Any]] = []
    loaded_games = 0
    skipped_oversized = 0
    timestamp_regressions = 0
    scanned_lines = 0
    with gzip.open(path, "rb") as handle:
        for source_line, line in bounded_jsonl_lines(handle, MAX_GAME_LINE_BYTES):
            scanned_lines = source_line
            if source_line > MAX_SOURCE_LINES or loaded_games >= MAX_REPLAY_GAMES:
                break
            if line is None:
                skipped_oversized += 1
                continue
            if not line.strip():
                continue
            game = json.loads(line.decode("utf-8"))
            events = game.get("events") if isinstance(game, dict) else None
            if not isinstance(events, list):
                raise ValueError(f"dataset line {source_line}: events must be an array")
            game_index = loaded_games
            loaded_games += 1
            packet_limit = min(len(events), MAX_PACKETS_PER_GAME)
            ticks, packet_times, game_timestamp_regressions = packet_ticks(events, game_index, packet_limit)
            timestamp_regressions += game_timestamp_regressions
            sampled = [
                (packet_index, ticks[packet_index], packet_times[packet_index], compact_event(events[packet_index], game_index, packet_index))
                for packet_index in range(0, packet_limit, PACKET_SAMPLE_STRIDE)
            ]
            stable_source = f"{HF_DATASET_REPO}@{HF_DATASET_REVISION}:{HF_REPLAY_FILE}:{source_line}"
            match_id = "hf-replay-" + hashlib.sha256(stable_source.encode("utf-8")).hexdigest()[:16]
            for (packet_index, tick, packet_time, packet), (next_index, next_tick, next_packet_time, next_packet) in zip(sampled, sampled[1:]):
                evidence_id = f"{dataset_url}/blob/{HF_DATASET_REVISION}/{HF_REPLAY_FILE}#line-{source_line}-packet-{next_index}"
                converted.append({
                    "matchId": match_id,
                    "tick": tick,
                    "state": {"task": "next_sampled_packet_prediction", "packetIndex": packet_index, "sampleStride": PACKET_SAMPLE_STRIDE, "currentPacket": packet},
                    "label": {"analysis": json.dumps({"nextSampledPacket": next_packet}, sort_keys=True, separators=(",", ":")), "task": "next_sampled_packet_prediction", "sourceType": "licensed_replay_export", "evidenceId": evidence_id},
                    "metadata": {
                        "task": "next_sampled_packet_prediction", "datasetRepo": HF_DATASET_REPO,
                        "datasetRevision": HF_DATASET_REVISION, "datasetFile": HF_REPLAY_FILE,
                        "datasetFileSha256": HF_REPLAY_SHA256, "datasetLicense": "Apache-2.0 (publisher-declared)",
                        "datasetUrl": dataset_url, "sourceLine": source_line, "packetTimeSeconds": packet_time,
                        "targetTick": next_tick, "targetPacketIndex": next_index, "targetPacketTimeSeconds": next_packet_time,
                        "coordinateMethod": "dataset-native x/z packet coordinates; not normalized simulator coordinates",
                        "uncertainty": "decoded packet sequence target; no coaching or player-intent label",
                    },
                })
    if not converted:
        raise ValueError("The selected batch and safety limits produced no packet transitions")
    return converted, {"scannedLines": scanned_lines, "loadedGames": loaded_games, "skippedOversizedGames": skipped_oversized, "timestampRegressions": timestamp_regressions}

records, conversion_report = records_from_replay_batch(replay_batch_path)
partial_records = OUTPUT_RECORDS.with_name(OUTPUT_RECORDS.name + ".part")
with partial_records.open("w", encoding="utf-8") as handle:
    for record in records:
        handle.write(json.dumps(record, ensure_ascii=False, separators=(",", ":")) + "\n")
partial_records.replace(OUTPUT_RECORDS)
manifest = {
    "schemaVersion": "1.1",
    "dataset": {"repo": HF_DATASET_REPO, "revision": HF_DATASET_REVISION, "file": HF_REPLAY_FILE, "sha256": HF_REPLAY_SHA256, "license": "Apache-2.0 (publisher-declared)"},
    "limits": {"maxSourceLines": MAX_SOURCE_LINES, "maxGameLineBytes": MAX_GAME_LINE_BYTES, "maxReplayGames": MAX_REPLAY_GAMES, "maxPacketsPerGame": MAX_PACKETS_PER_GAME, "packetSampleStride": PACKET_SAMPLE_STRIDE},
    "result": {**conversion_report, "records": len(records), "recordsSha256": sha256_file(OUTPUT_RECORDS)},
    "disclosure": "Offline decoded packet transitions; no coaching labels, player-intent claims, normalized simulator coordinates, or runtime telemetry.",
}
OUTPUT_MANIFEST.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))
print(f"Wrote prepared records: {OUTPUT_RECORDS.resolve()}")



{
"e
ven
ts":
[{"Cr
eateHe
ro":{"t
ime":0.0
,"net_id"
:107374185
4,"name":"g
g jg gap","c
hampion":"ren
gar"}},{"DoSet
Cooldown":{"tim
e":0.0,"net_id":
1073741854,"slot"
:0,"cooldown":0.0,
"display_cooldown":
-1.0}},{"DoSetCooldo
wn":{"time":0.0,"net_
id":1073741854,"slot":
1,"cooldown":0.0,"displ
ay_cooldown":-1.0}},{"Do
SetCooldown":{"time":0.0,
"net_id":1073741854,"slot"
:2,"cooldown":0.0,"display_
cooldown":-1.0}},{"DoSetCool
down":{"time":0.0,"net_id":10
73741854,"slot":3,"cooldown":0
.0,"display_cooldown":-1.0}},{"
DoSetCooldown":{"time":0.0,"net_
id":1073741854,"slot":4,"cooldown
":0.0,"display_cooldown":-1.0}},{"
DoSetCooldown":{"time":0.0,"net_id"
:1073741854,"slot":5,"cooldown":0.0,
"display_cooldown":-1.0}},{"DoSetCool
down":{"time":0.0,"net_id":1073741854,
"slot":13,"cooldown":0.0,"display_coold
own":-1.0}},{"DoSetCooldown":{"time":0.0
,"net_id":1073741854,"slot":45,"cooldown"
:0.0,"display_cooldown":-1.0}},{"DoSetCool
down":{"time":0.0,"net_id":1073741854,"slot
":46,"cool

In [ ]:
def show_raw_packets_file(
    path: Path = replay_batch_path, *, game_index: int = 0, start_packet: int = 0, limit: int = 5
) -> int:
    """Pretty-print a bounded packet range from one game in the raw gzip JSONL batch."""
    for name, value in (("game_index", game_index), ("start_packet", start_packet)):
        if not isinstance(value, int) or isinstance(value, bool) or value < 0:
            raise ValueError(f"{name} must be a non-negative integer")
    if not isinstance(limit, int) or isinstance(limit, bool) or not 1 <= limit <= 20:
        raise ValueError("limit must be an integer between 1 and 20")
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Raw packet file not found: {path.resolve()}")

    selected_events: list[Any] | None = None
    selected_source_line: int | None = None
    current_game_index = -1
    with gzip.open(path, "rb") as handle:
        for source_line, line in bounded_jsonl_lines(handle, MAX_GAME_LINE_BYTES):
            if line is None or not line.strip():
                continue
            current_game_index += 1
            if current_game_index != game_index:
                continue
            game = json.loads(line.decode("utf-8"))
            events = game.get("events") if isinstance(game, dict) else None
            if not isinstance(events, list):
                raise ValueError(f"Raw game on source line {source_line} has no events array")
            selected_events = events
            selected_source_line = source_line
            break

    if selected_events is None:
        raise IndexError(f"Raw packet file has no readable game at index {game_index}")
    print(f"File: {path.resolve()} ({path.stat().st_size:,} compressed bytes)")
    print(f"Game {game_index} (source line {selected_source_line}): {len(selected_events):,} packets")
    print("Warning: this is unredacted source data and may contain player or account identifiers.")
    shown = 0
    for packet_index in range(start_packet, min(len(selected_events), start_packet + limit)):
        print(f"\nPacket {packet_index}:")
        print(json.dumps(selected_events[packet_index], ensure_ascii=False, indent=2))
        shown += 1
    if shown == 0:
        print(f"No packets found at or after index {start_packet}.")
    return shown

def show_prepared_file(path: Path = OUTPUT_RECORDS, *, start: int = 0, limit: int = 3) -> int:
    """Pretty-print a bounded range of records from a prepared JSONL file."""
    if not isinstance(start, int) or isinstance(start, bool) or start < 0:
        raise ValueError("start must be a non-negative integer")
    if not isinstance(limit, int) or isinstance(limit, bool) or not 1 <= limit <= 20:
        raise ValueError("limit must be an integer between 1 and 20")
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Prepared JSONL file not found: {path.resolve()}")

    print(f"File: {path.resolve()} ({path.stat().st_size:,} bytes)")
    shown = 0
    with path.open("r", encoding="utf-8") as handle:
        for record_index, line in enumerate(handle):
            if record_index < start:
                continue
            if shown >= limit:
                break
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON on file line {record_index + 1}") from error
            print(f"\nRecord {record_index}:")
            print(json.dumps(record, ensure_ascii=False, indent=2))
            shown += 1
    if shown == 0:
        print(f"No records found at or after index {start}.")
    return shown

show_raw_packets_file(start_packet=2000, limit=20)

File: /content/playable-replays-output/hf-cache/04f9c7350e9ffcc689b731875ad9baf3ff6eaa6d/13_2/batch_001.jsonl.gz (86,983,155 compressed bytes)
Game 0 (source line 1): 270,201 packets

Packet 0:
{
  "CreateHero": {
    "time": 0.0,
    "net_id": 1073741854,
    "name": "gg jg gap",
    "champion": "rengar"
  }
}

Packet 1:
{
  "DoSetCooldown": {
    "time": 0.0,
    "net_id": 1073741854,
    "slot": 0,
    "cooldown": 0.0,
    "display_cooldown": -1.0
  }
}

Packet 2:
{
  "DoSetCooldown": {
    "time": 0.0,
    "net_id": 1073741854,
    "slot": 1,
    "cooldown": 0.0,
    "display_cooldown": -1.0
  }
}

Packet 3:
{
  "DoSetCooldown": {
    "time": 0.0,
    "net_id": 1073741854,
    "slot": 2,
    "cooldown": 0.0,
    "display_cooldown": -1.0
  }
}

Packet 4:
{
  "DoSetCooldown": {
    "time": 0.0,
    "net_id": 1073741854,
    "slot": 3,
    "cooldown": 0.0,
    "display_cooldown": -1.0
  }
}

Packet 5:
{
  "DoSetCooldown": {
    "time": 0.0,
    "net_id": 1073741854,
    "slot": 4,
   

20

## Handoff

1. Run this notebook on a Google Colab CPU runtime; local download is intentionally blocked.
2. Confirm the checksum and manifest output. Oversized games are reported and skipped.
3. On the same runtime, open [`player-ai.ipynb`](./player-ai.ipynb). Its default `DATA_PATH` points to the exported JSONL.
4. Inspect the training preview before enabling QLoRA.

Sources: [dataset card and packet schema](https://huggingface.co/datasets/maknee/league-of-legends-decoded-replay-packets), [pinned default batch and published checksum](https://huggingface.co/datasets/maknee/league-of-legends-decoded-replay-packets/blob/04f9c7350e9ffcc689b731875ad9baf3ff6eaa6d/12_22/batch_001.jsonl.gz).